## Lab6-Assignment: Topic Classification

Use the same training, development, and test partitions of the the 20 newsgroups text dataset as in Lab6.4-Topic-classification-BERT.ipynb 

* Fine-tune and examine the performance of another transformer-based pretrained language models, e.g., RoBERTa, XLNet

* Compare the performance of this model to the results achieved in Lab6.4-Topic-classification-BERT.ipynb and to a conventional machine learning approach (e.g., SVM, Naive Bayes) using bag-of-words or other engineered features of your choice. 
Describe the differences in performance in terms of Precision, Recall, and F1-score evaluation metrics.

## Assignment implementation: fine-tune RoBERTa for topic classification

In this section we fine-tune `roberta-base` on the same 4-topic subset of the 20 newsgroups dataset used in `Lab6.4-Topic-classification-BERT.ipynb`.

The notebook below installs the required packages, prepares train/dev/test splits, fine-tunes the model, and reports precision, recall, and F1-score on the test set.

In [7]:
!pip install -q pandas numpy scikit-learn simpletransformers torch


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\smart\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from simpletransformers.classification import ClassificationModel, ClassificationArgs

# Load the same 4 categories used in the BERT notebook
categories = ['alt.atheism', 'comp.graphics', 'sci.med', 'sci.space']

newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'), categories=categories, random_state=42)
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'), categories=categories, random_state=42)

train_df = pd.DataFrame({
    'text': newsgroups_train.data,
    'labels': newsgroups_train.target
})
test_df = pd.DataFrame({
    'text': newsgroups_test.data,
    'labels': newsgroups_test.target
})

train_df, dev_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=train_df[['labels']]
)

print('Train size:', len(train_df))
print('Dev size:', len(dev_df))
print('Test size:', len(test_df))
print('Labels:', train_df['labels'].value_counts().sort_index().to_dict())

c:\Users\keimp\miniconda3\envs\st_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train size: 2025
Dev size: 226
Test size: 1498
Labels: {0: 432, 1: 525, 2: 534, 3: 534}


In [2]:
model_args = ClassificationArgs()
model_args.overwrite_output_dir = True
model_args.evaluate_during_training = True
model_args.evaluate_during_training_steps = 32
model_args.save_eval_checkpoints = False
model_args.save_model_every_epoch = False
model_args.num_train_epochs = 3
model_args.train_batch_size = 16
model_args.eval_batch_size = 32
model_args.learning_rate = 4e-5
model_args.max_seq_length = 256
model_args.use_multiprocessing = False
model_args.use_multiprocessing_for_evaluation = False
model_args.no_cache = True

model = ClassificationModel(
    'roberta',
    'roberta-base',
    num_labels=4,
    args=model_args,
    use_cuda=False
)

model.train_model(train_df, eval_df=dev_df)

c:\Users\keimp\miniconda3\envs\st_env\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\keimp\.cache\huggingface\hub\models--roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download.

(381,
 defaultdict(list,
             {'global_step': [32,
               64,
               96,
               127,
               128,
               160,
               192,
               224,
               254,
               256,
               288,
               320,
               352,
               381],
              'train_loss': [0.675175666809082,
               0.45642009377479553,
               0.39936405420303345,
               0.3674660325050354,
               0.17560923099517822,
               0.2374097853899002,
               0.41238918900489807,
               0.29034340381622314,
               0.19367863237857819,
               0.24737395346164703,
               0.05593894049525261,
               0.14098651707172394,
               0.004387254826724529,
               0.0033343329560011625],
              'mcc': [np.float64(0.8099544165071254),
               np.float64(0.8307415540259239),
               np.float64(0.888180544840005),
               np

In [3]:
# Evaluate on the development set
result_dev, _, _ = model.eval_model(dev_df)
print('Dev evaluation results:')
print(result_dev)

# Predict on the test set
predictions, _ = model.predict(test_df['text'].tolist())
test_df['predicted'] = predictions

print('Test set classification report:')
print(classification_report(test_df['labels'], test_df['predicted'], target_names=newsgroups_train.target_names))

Running Evaluation: 100%|██████████| 8/8 [00:33<00:00,  4.22s/it]


Dev evaluation results:
{'mcc': np.float64(0.876453481568735), 'eval_loss': 0.28235851452336647}


Predicting: 100%|██████████| 47/47 [03:33<00:00,  4.55s/it]

Test set classification report:
               precision    recall  f1-score   support

  alt.atheism       0.83      0.82      0.83       319
comp.graphics       0.86      0.93      0.89       389
      sci.med       0.91      0.87      0.89       396
    sci.space       0.85      0.82      0.84       394

     accuracy                           0.86      1498
    macro avg       0.86      0.86      0.86      1498
 weighted avg       0.86      0.86      0.86      1498



# Assignment Implementation: Conventional SVM with Bag-of-Words Approach

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

## TF-IDF + Linear SVM implementation

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        sublinear_tf=True,   # Dampen frequent terms
        max_df=0.95,         # ignore terms that appear in 95% of documents
        min_df=2,            # ignore terms that appear in less than 2 documents
        ngram_range=(1, 2),  # unigrams + bigrams
        stop_words='english'
    )),

    ('svm', LinearSVC(
        C=1.0,
        max_iter=2000,
        random_state=42
    ))
])

# Train on the training split
pipeline.fit(train_df['text'], train_df['labels'])
print('Model trained.')

Model trained.


In [8]:
# SVM + Bag-of-Words evaluation

test_preds = pipeline.predict(test_df['text'])

print('Test Set Classification Report:')
report = classification_report(
    test_df['labels'],
    test_preds,
    target_names=newsgroups_train.target_names,
    output_dict=True
)
print(classification_report(
    test_df['labels'],
    test_preds,
    target_names=newsgroups_train.target_names
))

Test Set Classification Report:
               precision    recall  f1-score   support

  alt.atheism       0.83      0.78      0.80       319
comp.graphics       0.88      0.88      0.88       389
      sci.med       0.90      0.83      0.86       396
    sci.space       0.77      0.86      0.81       394

     accuracy                           0.84      1498
    macro avg       0.84      0.84      0.84      1498
 weighted avg       0.84      0.84      0.84      1498



## BERT Evaluation Report

                        precision    recall  f1-score   support

         alt.atheism       0.85      0.82      0.83       319
         comp.graphics     0.90      0.92      0.91       389
         sci.med           0.86      0.92      0.89       396
         sci.space         0.87      0.82      0.84       394

         accuracy                              0.87      1498
         macro avg         0.87      0.87      0.87      1498
         weighted avg      0.87      0.87      0.87      1498

## RoBERTa Evaluation Report
                        precision    recall  f1-score   support

         alt.atheism       0.83      0.82      0.83       319
         comp.graphics     0.86      0.93      0.89       389
         sci.med           0.91      0.87      0.89       396
         sci.space         0.85      0.82      0.84       394

         accuracy                              0.86      1498
         macro avg         0.86      0.86      0.86      1498
         weighted avg      0.86      0.86      0.86      1498

## SVM + BoW Evaluation Report
                        precision    recall  f1-score   support

         alt.atheism       0.83      0.78      0.80       319
         comp.graphics     0.88      0.88      0.88       389
         sci.med           0.90      0.83      0.86       396
         sci.space         0.77      0.86      0.81       394

         accuracy                              0.84      1498
         macro avg         0.84      0.84      0.84      1498
         weighted avg      0.84      0.84      0.84      1498

## Comparing the results



- The three models show relatively strong performance, but the transformer-based models perform better overall than the conventional SVM + Bag-of-Words approach.

- BERT achieves the best overall performance, with an accuracy, macro average F1-score, and weighted average F1-score of 0.87. RoBERTa performs very similarly, with an accuracy and macro/weighted F1-score of 0.86. The SVM + BoW model performs slightly lower, reaching 0.84 for accuracy, macro average F1-score, and weighted average F1-score.

- The difference between BERT and RoBERTa is small. BERT performs slightly better on comp.graphics, with an F1-score of 0.91 compared to RoBERTa's 0.89. Both models perform equally well on alt.atheism and sci.space, while RoBERTa and BERT both reach an F1-score of 0.89 on sci.med.

- Compared to the transformer models, the SVM model performs worse on most categories. Its weakest class is alt.atheism, with an F1-score of 0.80, and it also scores lower on sci.space with an F1-score of 0.81. However, the SVM model still performs well overall

- The transformer models likely perform better because they use pretrained contextual embeddings, whereas the SVM + BoW approach relies mainly on word frequency patterns and does not understand context in the same way. 

- Overall, BERT gives the strongest results, RoBERTa is very close behind, and the SVM + BoW model provides a strong but less accurate baseline. The results show that transformer-based models are more effective for topic classification, although traditional machine learning methods can still perform competitively on this dataset.


### Notes

* This notebook fine-tunes `roberta-base` instead of `bert-base-cased`.
* The key performance metrics are Precision, Recall and F1-score on the held-out test set.
* To complete the full assignment, compare these results to the BERT results in `Lab6.4-Topic-classification-BERT.ipynb` and to a conventional baseline such as SVM or Naive Bayes with bag-of-words features.